# KUNCI JAWABAN — Aktivitas Naive Bayes (Play Tennis)

**File ini untuk dosen.** Jangan bagikan ke mahasiswa sebelum sesi selesai.

Catatan singkat untuk dosen:
- Perhitungan manual menggunakan rumus polos *tanpa* Laplace smoothing (sesuai slide 2.2).
- `CategoricalNB` sklearn default pakai `alpha=1` (Laplace), jadi probabilitasnya akan **berbeda** dari manual — meski **kelas prediksinya sama** (No). Ini sengaja, untuk memicu diskusi pada Tahap 4 nomor 1–2.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option("display.precision", 4)
np.set_printoptions(precision=4, suppress=True)

## Tahap 1 — Memahami Data

In [ ]:
url = "https://raw.githubusercontent.com/USERNAME/naive-bayes/main/data/play_tennis.csv"
df = pd.read_csv(url)
df

In [ ]:
df["Play"].value_counts()
# Output yang diharapkan: Yes=9, No=5

## Tahap 2 — Hitung Manual

Kasus uji: Outlook=Sunny, Temperature=Cool, Humidity=High, Wind=Strong

In [ ]:
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}

### 2.1 Prior

In [ ]:
n_total = len(df)
n_yes = (df["Play"] == "Yes").sum()
n_no  = (df["Play"] == "No").sum()

p_yes = n_yes / n_total
p_no  = n_no  / n_total

print(f"P(Yes) = {p_yes:.4f}   (= 9/14)")
print(f"P(No)  = {p_no:.4f}   (= 5/14)")

### 2.2 Probabilitas Kondisional

In [ ]:
df_yes = df[df["Play"] == "Yes"]
df_no  = df[df["Play"] == "No"]

# Kelas Yes (denominator = 9)
p_sunny_yes  = (df_yes["Outlook"]     == "Sunny" ).sum() / len(df_yes)   # 2/9
p_cool_yes   = (df_yes["Temperature"] == "Cool"  ).sum() / len(df_yes)   # 3/9
p_high_yes   = (df_yes["Humidity"]    == "High"  ).sum() / len(df_yes)   # 3/9
p_strong_yes = (df_yes["Wind"]        == "Strong").sum() / len(df_yes)   # 3/9

# Kelas No (denominator = 5)
p_sunny_no  = (df_no["Outlook"]     == "Sunny" ).sum() / len(df_no)      # 3/5
p_cool_no   = (df_no["Temperature"] == "Cool"  ).sum() / len(df_no)      # 1/5
p_high_no   = (df_no["Humidity"]    == "High"  ).sum() / len(df_no)      # 4/5
p_strong_no = (df_no["Wind"]        == "Strong").sum() / len(df_no)      # 3/5

print("Yes:", p_sunny_yes, p_cool_yes, p_high_yes, p_strong_yes)
print("No :", p_sunny_no,  p_cool_no,  p_high_no,  p_strong_no)

### 2.3 Posterior & Keputusan

In [ ]:
score_yes = p_yes * p_sunny_yes * p_cool_yes * p_high_yes * p_strong_yes
score_no  = p_no  * p_sunny_no  * p_cool_no  * p_high_no  * p_strong_no

total = score_yes + score_no
post_yes = score_yes / total
post_no  = score_no  / total

prediksi_manual = "Yes" if post_yes > post_no else "No"

print(f"score(Yes) = {score_yes:.6f}")
print(f"score(No)  = {score_no:.6f}")
print(f"P(Yes | x) = {post_yes:.4f}")
print(f"P(No  | x) = {post_no:.4f}")
print(f"Prediksi manual: {prediksi_manual}")
# Hasil yang diharapkan: P(Yes|x) ~ 0.2046, P(No|x) ~ 0.7954, prediksi = No

## Tahap 3 — Implementasi sklearn

In [ ]:
features = ["Outlook", "Temperature", "Humidity", "Wind"]

encoder = OrdinalEncoder()
X = encoder.fit_transform(df[features])
y = df["Play"].values

for col, cats in zip(features, encoder.categories_):
    print(f"  {col}: {dict(enumerate(cats))}")

In [ ]:
model = CategoricalNB()      # alpha=1 (Laplace smoothing) secara default
model.fit(X, y)

y_pred = model.predict(X)
akurasi = accuracy_score(y, y_pred)
print(f"Akurasi pada data latih: {akurasi:.4f}")

In [ ]:
x_test_df = pd.DataFrame([test])
x_test = encoder.transform(x_test_df[features])

prediksi_sklearn = model.predict(x_test)
proba_sklearn = model.predict_proba(x_test)

print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")
# Diharapkan: prediksi=No; probabilitas ~ [0.7196, 0.2804] (urutan kelas No, Yes)

## Tahap 4 — Pembahasan Diskusi (untuk dosen)

**1. Apakah prediksi & probabilitas sama?**  
Prediksi sama (`No`). Probabilitas berbeda: manual ≈ (0.2046, 0.7954) sedangkan sklearn ≈ (0.2804, 0.7196).  
Penyebab: `CategoricalNB` default `alpha=1` (Laplace smoothing). Setiap probabilitas kondisional dihitung sebagai:
$$P(F_i = t \mid K) = \frac{N_{tic} + \alpha}{N_c + \alpha \cdot n_i}$$
di mana $n_i$ = jumlah kategori unik fitur ke-i. Smoothing menggeser probabilitas ke arah uniform.

**2. Eksperimen `alpha=1e-10`** — lihat sel di bawah.

**3. Smoothing & kategori yang hilang.**  
Tanpa smoothing, jika sebuah kombinasi (kategori, kelas) frekuensinya 0, maka P(Fi|K)=0 — perkalian posterior jadi 0 dan kelas itu *tidak akan pernah* dipilih meskipun fitur lain mendukung. Laplace smoothing menambahkan +α sehingga probabilitas tidak pernah persis 0.

**4. Independensi fitur.**  
Tidak benar-benar independen di dunia nyata: cuaca `Sunny` cenderung punya `Humidity` tertentu, `Outlook=Rain` korelasi dengan `Humidity=High`, dll. Konsekuensi: probabilitas posterior menjadi *over-confident* (terlalu yakin), tapi *peringkat* kelas seringkali masih benar — itulah kenapa Naive Bayes tetap akurat dalam praktek meskipun asumsinya cacat.

In [ ]:
# Demo nomor 2: alpha kecil mendekati hasil manual
model_no_smooth = CategoricalNB(alpha=1e-10)
model_no_smooth.fit(X, y)
proba_no_smooth = model_no_smooth.predict_proba(x_test)

print(f"Manual              : P(No|x)={post_no:.4f}, P(Yes|x)={post_yes:.4f}")
print(f"sklearn alpha=1     : {proba_sklearn[0]}  (kelas: {model.classes_})")
print(f"sklearn alpha=1e-10 : {proba_no_smooth[0]}  (kelas: {model_no_smooth.classes_})")